In [4]:
import sys
import json
import csv
import yaml

import importlib

import math

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime
# with the above choices, the imported datetime.time(2023,07,01) is recognized
# from datetime import date
# from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

# Create an utilities file util.py in a folder benchmarking and import it
# NOTE: I moved my util.py to the directory "helper_functions" -- seems like a better name
sys.path.append('../ECS116-HELPER-FUNCTIONS/')
import util

# checking that my util.py file is accessible
util.hello_world()

'hello world'

In [7]:
from pymongo import MongoClient

client = MongoClient()
# could have written client = MongoClient("localhost", 27017)
#                 or client = MongoClient("mongodb://localhost:27017/")

db = client.airbnb

print('The list of all databases currently in the MongoDB client is:')
print(client.list_database_names())

print('\nThe list of all collections in the airbnb database is:')
print(db.list_collection_names())

The list of all databases currently in the MongoDB client is:
['admin', 'airbnb', 'config', 'local', 'test']

The list of all collections in the airbnb database is:
['listings_small', 'listings_test', 'listings_with_calendar', 'listings_with_reviews', 'calendar', 'listings_with_reviews_duplicate']


In [8]:
filenamel = '/Users/rick/DM-for-DS-2025/DATA-SETS/AirBNB/New-York-City/listings.csv'

# Using partial list of dtypes, so that first several fields are interpreted as strings
# As for the date and available fields (intended as date type and boolean, respectively,
#    we import as strings and convert in the data frame
dtype = {"id": str,  "host_id": str }
# note including these, because the null values make trouble:  , "minimum_nights": int, "maximum_nights": int}

# the csv has nulls in "adjusted_price", which has type str,. so including keep_default_na=False, 
#    see https://stackoverflow.com/questions/10867028/get-pandas-read-csv-to-read-empty-values-as-empty-string-instead-of-nan
         
df_listings = pd.read_csv(filenamel, dtype=dtype, keep_default_na=False)

/var/folders/3l/gd1qj_mw8xl3gm001s6w9h180000gp/T/ipykernel_82687/409428695.py:12: DtypeWarning: Columns (43,44,45,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listings = pd.read_csv(filenamel, dtype=dtype, keep_default_na=False)


In [9]:
print(df_listings.shape)

(37434, 79)


<span style=color:blue>Dropping several columns from df_listings, retaining only the 18 columns used in Part 1     </span>

In [10]:
print(df_listings.columns)

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [22]:
cols_to_drop = ['listing_url', 'scrape_id', 'last_scraped', 'source',
        'description', 'neighborhood_overview', 'picture_url', 
       'host_url', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'property_type', 
       'maximum_nights', 'minimum_minimum_nights',
       'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'calendar_updated', 
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'calendar_last_scraped',
       'number_of_reviews_l30d', 'availability_eoy',
       'number_of_reviews_ly', 'estimated_occupancy_l365d',
       'estimated_revenue_l365d', 'first_review',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'instant_bookable',
       'calculated_host_listings_count_entire_homes',
       'calculated_host_listings_count_private_rooms',
       'calculated_host_listings_count_shared_rooms', 
            'accommodates', 'bathrooms', 'bathrooms_text' ,
                'bedrooms', 'beds', 'amenities',
            'review_scores_rating', 'review_scores_accuracy'
]

df_listings.drop(cols_to_drop, axis=1, inplace=True)
print(df_listings.shape)

print(df_listings.columns)





(37434, 18)
Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude', 'room_type',
       'price', 'minimum_nights', 'has_availability', 'number_of_reviews',
       'number_of_reviews_ltm', 'last_review', 'license',
       'calculated_host_listings_count', 'reviews_per_month'],
      dtype='object')


In [23]:
filenamer = '/Users/rick/DM-for-DS-2024/DATA-SETS/AirBNB/New-York-City/reviews.csv'

# Using partial list of dtypes, so that first several fields are interpreted as strings
# As for the date and available fields (intended as date type and boolean, respectively,
#    we import as strings and convert in the data frame
dtype = {"id": str, "listing_id": str, "reviewer_id": str}
# note including these, because the null values make trouble:  , "minimum_nights": int, "maximum_nights": int}

# the csv has nulls in "adjusted_price", which has type str,. so including keep_default_na=False, 
#    see https://stackoverflow.com/questions/10867028/get-pandas-read-csv-to-read-empty-values-as-empty-string-instead-of-nan
         
df_reviews = pd.read_csv(filenamer, dtype=dtype, keep_default_na=False)

In [24]:
print(df_reviews.shape)

(986810, 6)


In [30]:
# also put this into util.py
# dt has format such as '1/3/24'
def convert_date_dash_to_datetime(dt):
    if dt is None:
        return None
    elif pd.isnull(dt):  # tests whether dt is the pandas value NaT ("not a time")
        # print('\nEntered the NaT case\n')
        return None
    elif dt != dt:
        return None        # could also use math.nan, I think
    elif dt == '':
        return None
    else:
        # dt = dt[:-2] + '20' + dt[-2:]
        new_dt = datetime.strptime(dt, '%Y-%m-%d') 
        return new_dt

print(convert_date_slash_to_datetime('1/3/23'))

2023-01-03 00:00:00


In [31]:
# using this because I added stuff to util.py
# importlib.reload(util)

# dates in listings.csv have format such as '1/3/23', so using function to convert that to datetime
df_listings['last_review'] = df_listings['last_review'].apply(convert_date_dash_to_datetime)

In [34]:
df_listings['price'] = pd.to_numeric(df_listings['price'].str.replace('$','').str.replace(',',''))

In [35]:
df_listings['reviews_per_month'] = pd.to_numeric(df_listings['reviews_per_month'])

In [36]:
print(df_listings.dtypes)

id                                        object
name                                      object
host_id                                   object
host_name                                 object
neighbourhood_cleansed                    object
neighbourhood_group_cleansed              object
latitude                                 float64
longitude                                float64
room_type                                 object
price                                    float64
minimum_nights                             int64
has_availability                          object
number_of_reviews                          int64
number_of_reviews_ltm                      int64
last_review                       datetime64[ns]
license                                   object
calculated_host_listings_count             int64
reviews_per_month                        float64
dtype: object


In [38]:
df_reviews['date'] = df_reviews['date'].apply(convert_date_dash_to_datetime)

In [39]:
print(df_reviews.dtypes)

listing_id               object
id                       object
date             datetime64[ns]
reviewer_id              object
reviewer_name            object
comments                 object
dtype: object


In [40]:
dict_listings = df_listings.to_dict('records')
print(len(dict_listings))

37434


In [41]:
dict_reviews = df_reviews.to_dict('records')
print(len(dict_reviews))

986810


In [42]:
# converting the NaT's into Nones
i = 0
for d in dict_listings:
    if pd.isnull(d['last_review']):
        i = i+1
        d['last_review'] = None
print(i)

11787


In [43]:
db.listings_3.drop()

time1 = datetime.now()
result = db.listings_3.insert_many(dict_listings)
time2 = datetime.now()
print(f'Time to load into MongoDB was {util.time_diff(time1,time2)} seconds.')

Time to load into MongoDB was 0.628229 seconds.


In [44]:
print(db.listings_3.count_documents({}))

37434


In [45]:
db.reviews_3.drop()

time1 = datetime.now()
result = db.reviews_3.insert_many(dict_reviews)
time2 = datetime.now()
print(f'Time to load into MongoDB was {util.time_diff(time1,time2)} seconds.')
print()
print(db.reviews_3.count_documents({}))

Time to load into MongoDB was 13.976149 seconds.

986810


<span style=color:blue> Adding the index    </span>

In [46]:
time1 = datetime.now()
index_name = db.reviews_3.create_index('listing_id')
time2 = datetime.now()
print(f'The time taken to create the index was {util.time_diff(time1,time2)} seconds.')
print(index_name)

The time taken to create the index was 1.975859 seconds.
listing_id_1


In [66]:
cursor = db.listings_with_reviews_m_3.find({'id' : {'$regex' : '^1001.*$'}})
    
l = list(cursor)
print(len(l))

28


In [47]:
db.listings_with_reviews_m_3.drop()

pipeline = [
    {
        '$lookup': {
            'from': 'reviews_3',
            'localField': 'id',
            'foreignField': 'listing_id',
            'as': 'reviews'
        }
    },
    {
        '$out': 'listings_with_reviews_m_3'
    }
]


time1 = datetime.now()
print(time1)
db.listings_3.aggregate(pipeline)
time2 = datetime.now()
print(f'Time to load into MongoDB was {util.time_diff(time1,time2)} seconds.')

print(db.listings_with_reviews_m_3.count_documents({}))


2025-05-21 17:57:18.260393
Time to load into MongoDB was 5.719389 seconds.
37434


In [48]:
doc = db.listings_with_reviews_m_3.find_one()
pprint.pp(doc)

{'_id': ObjectId('682e75e57297e60d4a175744'),
 'id': '36121',
 'name': 'Lg Rm in Historic Prospect Heights',
 'host_id': '62165',
 'host_name': 'Michael',
 'neighbourhood_cleansed': 'Prospect Heights',
 'neighbourhood_group_cleansed': 'Brooklyn',
 'latitude': 40.67376,
 'longitude': -73.96611,
 'room_type': 'Private room',
 'price': 200.0,
 'minimum_nights': 90,
 'has_availability': 't',
 'number_of_reviews': 9,
 'number_of_reviews_ltm': 0,
 'last_review': datetime.datetime(2013, 5, 10, 0, 0),
 'license': '',
 'calculated_host_listings_count': 1,
 'reviews_per_month': 0.05,
 'reviews': [{'_id': ObjectId('682e75fc7297e60d4a185b98'),
              'listing_id': '36121',
              'id': '152196',
              'date': datetime.datetime(2010, 12, 11, 0, 0),
              'reviewer_id': '240039',
              'reviewer_name': 'Nikki',
              'comments': 'Michael is the man! Funny, smart and brilliant. His '
                          'place is awesome, in a cool neighborhood and 

In [49]:
def convert_lwr_to_json(doc):
    doc_new = {}
    # start by transferring all scalar keys over, then fix some of them
    for key in doc.keys():
        if key != 'reviews':
            doc_new[key] = doc[key]
    # now fixing some possible issues
    doc_new['_id'] = str(doc['_id'])
    if doc['last_review'] == None:    # is null
        doc_new['last_review'] = None
    else:
        doc_new['last_review'] = doc['last_review'].strftime('%Y-%m-%d')
    if math.isnan(doc['price']):
        doc_new['price'] = None
    else:
        doc_new['price'] = doc['price']
    if math.isnan(doc['reviews_per_month']):
        doc_new['reviews_per_month'] = None
    else:
        doc_new['reviews_per_month'] = doc['reviews_per_month']
    # now dealing with the 'review' array
    dlist = []
    for d in doc['reviews']:
        d_new = {}
        d_new['_id'] = str(d['_id'])
        d_new['date'] = d['date'].strftime('%Y-%m-%d')
        for key in d.keys():
            if key not in ['date', '_id']:
                d_new[key] = d[key]
        dlist.append(d_new)
    doc_new['reviews'] = dlist
    return doc_new

# pprint.pp(doc)

pprint.pp(convert_lwr_to_json(doc))

{'_id': '682e75e57297e60d4a175744',
 'id': '36121',
 'name': 'Lg Rm in Historic Prospect Heights',
 'host_id': '62165',
 'host_name': 'Michael',
 'neighbourhood_cleansed': 'Prospect Heights',
 'neighbourhood_group_cleansed': 'Brooklyn',
 'latitude': 40.67376,
 'longitude': -73.96611,
 'room_type': 'Private room',
 'price': 200.0,
 'minimum_nights': 90,
 'has_availability': 't',
 'number_of_reviews': 9,
 'number_of_reviews_ltm': 0,
 'last_review': '2013-05-10',
 'license': '',
 'calculated_host_listings_count': 1,
 'reviews_per_month': 0.05,
 'reviews': [{'_id': '682e75fc7297e60d4a185b98',
              'date': '2010-12-11',
              'listing_id': '36121',
              'id': '152196',
              'reviewer_id': '240039',
              'reviewer_name': 'Nikki',
              'comments': 'Michael is the man! Funny, smart and brilliant. His '
                          'place is awesome, in a cool neighborhood and easy '
                          'to get around Brooklyn or to Manhat

In [65]:
print(db.listings_test.count_documents({}))

cursor = db.listings_test.find({'id' : {'$regex' : '^1001.*$'}})
    
l = list(cursor)
print(len(l))

37434
28


In [67]:
print(db.listings_with_reviews.count_documents({}))

cursor = db.listings_with_reviews_m_3.find({'id' : {'$regex' : '^1001.*$'}})
    
l = list(cursor)
print(len(l))

37434
28


In [68]:
cursor = db.listings_with_reviews_m_3.find({'id' : {'$regex' : '^1001.*$'}})

output = []

for doc in cursor:
    output.append(convert_lwr_to_json(doc))

print(len(output))

28


In [69]:
# Writing dict to a json file into a json file in a subdirectory
# Also putting this function into my util.py

def write_dict_to_dir_json(dict, dir, filename):
    with open(dir  + filename, 'w') as fp:
        json.dump(dict, fp)

dir = '/Users/rick/DM-for-DS-2025/PA3-OUTPUT/'
filename = 'listings_with_reviews_m_subset_1001.json'
write_dict_to_dir_json(output, dir, filename)

In [76]:
print(db.listings_with_reviews_m_3.count_documents({}))

cursor = db.listings_with_reviews_m_3.find({
    'number_of_reviews' : {
        '$gte' : 600
    }
})
    
l = list(cursor)
print(len(l))

37434
37


In [78]:
print()
cursor = db.listings_with_reviews_m_3.find({
    'number_of_reviews' : {
        '$gte' : 600
    }
})

output = []

for doc in cursor:
    output.append(convert_lwr_to_json(doc))

print(len(output))

dir = '/Users/rick/DM-for-DS-2025/PA3-OUTPUT/'
filename = 'listings_with_reviews_subset_number_of_reviews_600.json'
write_dict_to_dir_json(output, dir, filename)


37


In [72]:
dir = 'OUTPUTS'
filename = 'listings_with_reviews_m_subset_11110__v03.json'
util.write_dict_to_dir_json(output, dir, filename)

In [62]:
print(db.listings_with_reviews.count_documents({}))

cursor = db.listings_with_reviews_m_3.find({'id' : {'$regex' : '^1000.*$'}})
    
l = list(cursor)
print(len(l))

39202
43


In [63]:
cursor = db.listings_with_reviews_m_3.find({'id' : {'$regex' : '^1000.*$'}})

output = []

for doc in cursor:
    output.append(convert_lwr_to_json(doc))

print(len(output))

43


In [67]:
dir = 'OUTPUTS'
filename = 'listings_with_reviews_m_subset_1000.json'
util.write_dict_to_dir_json(output, dir, filename)

In [73]:
cursor = db.listings_with_reviews_m_3.find({})

output = []

for doc in cursor:
    output.append(convert_lwr_to_json(doc))

print(len(output))

39202


In [74]:
dir = 'OUTPUTS'
filename = 'listings_with_reviews_m_all__v03.json'
util.write_dict_to_dir_json(output, dir, filename)